# Data Loading Tutorial

This tutorial covers how to load EEG data from various formats supported by NeuRodent.

## Overview

NeuRodent supports multiple data formats commonly used in rodent EEG research:

1. **Binary files** (`.bin`) - Custom binary format
2. **SpikeInterface recordings** - Via the SpikeInterface library
3. **MNE objects** - From the MNE-Python library
4. **Neuroscope/Neuralynx** (`.dat`, `.eeg`)
5. **Open Ephys** (`.continuous`)
6. **NWB files** (`.nwb`) - Neurodata Without Borders format

The `LongRecordingOrganizer` class handles loading and organizing recordings from these formats.

## Setup

In [1]:
import sys
from pathlib import Path
import logging

from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

from neurodent import core
from neurodent import constants

import mne
import spikeinterface.core as si
import spikeinterface.extractors as se


# Set up logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(message)s", 
    level=logging.INFO
)
logger = logging.getLogger()

/mnt/isilon/marsh_single_unit/YY_PyEEG/neurodent_Yong/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Loading Binary Files

Binary files are a common format for storing continuous EEG data. NeuRodent can load binary files with associated metadata.

In [2]:
# Example: Loading from binary files
# Using included test data
data_path = Path("../../notebooks/tests/test-data/A10 KO 12_13_2023")
animal_id = "A10 KO 12_13_2023"

# Create LongRecordingOrganizer
# mode options: 'bin', 'si' (SpikeInterface), 'mne', etc.
lro = core.LongRecordingOrganizer(
    base_folder_path=data_path,
    animal_id=animal_id,
    mode="bin",  # Change based on your data format
)

print(f"Loaded recordings for {animal_id}")
print(f"Number of recordings: {len(lro.colbins)}")
print(f"Sampling frequency: {lro.meta.f_s}")
print(f"Number of channels: {lro.meta.n_channels}")

2026-01-30 16:20:54,964 - INFO - CSV metadata timestamps: 1 of 1 files have timestamps
2026-01-30 16:20:54,965 - INFO - Converting 1 column-major binary files to row-major format
2026-01-30 16:20:54,966 - INFO - Overwrite flag not set - only generating missing row-major files
2026-01-30 16:20:55,115 - INFO - Recording already at target sampling rate (1000 Hz), no resampling needed
2026-01-30 16:20:55,116 - INFO - LongRecording created: LongRecording: 1 files, 10 channels, 1000.0 Hz, 120.4s duration, channels: [Intan Input (1)/PortC C-009, Intan Input (1)/PortC C-010, Intan Input (1)/PortC C-012, Intan Input (1)/PortC C-014, Intan Input (1)/PortC C-015, Intan Input (1)/PortC C-016, Intan Input (1)/PortC C-017, Intan Input (1)/PortC C-019, Intan Input (1)/PortC C-021, Intan Input (1)/PortC C-022], float32 precision, µV units, timestamps: 1/1 files have timestamps
2026-01-30 16:20:55,118 - INFO - Finalizing file timestamps
2026-01-30 16:20:55,120 - INFO - Using CSV metadata timestamps


Loaded recordings for A10 KO 12_13_2023
Number of recordings: 1
Sampling frequency: 1000
Number of channels: 10


In [ ]:
# Include converting channel names

In [ ]:
lro_mne = lro.convert_to_mne()

for i in range(len(lro_mne.info['ch_names'])):
    mne_chname = lro_mne.info['ch_names'][i]
    lro_mne.info['ch_names'][i] = core.utils.parse_chname_to_abbrev(channel_name = mne_chname, assume_from_number=True, strict_matching=False)

mne.export.export_raw("../../notebooks/tests/test-data/A10 KO 12_13_2023/A10_si.edf", lro_mne, fmt="edf")
#lro_mne.save("../../notebooks/tests/test-data/A10 KO 12_13_2023/A10_mne.fif", overwrite=True)

# lro_mne.info['sfreq'] = int(lro.meta.f_s)
#print(lro_mne.info)

### Binary File Format Details

NeuRodent supports two binary layouts:

1. **Column-major** (default): Each file contains data for one channel
2. **Row-major**: Each file contains all channels, with samples interleaved

You can convert between formats:

In [ ]:
meta_path = "../../notebooks/tests/test-data/A10 KO 12_13_2023/Cage 2 A10-0_Meta.csv"
metadata = core.DDFBinaryMetadata(metadata_path=meta_path)

# Convert column-major to row-major format
col_path = "../../notebooks/tests/test-data/A10 KO 12_13_2023/Cage 2 A10-0_ColMajor.bin"
row_out_path = "../../notebooks/tests/test-data/A10 KO 12_13_2023"

core.convert_ddfcolbin_to_ddfrowbin(
    colbin_path=col_path,
    rowdir_path=row_out_path,
    metadata = metadata
)

# Convert row-major file to SpikeInterface format
# The output will be tuple with SpikeInterface recording and temporary file path
row_path = "../../notebooks/tests/test-data/A10 KO 12_13_2023/Cage 2 A10-0_RowMajor.npy.gz"
conv_si = core.convert_ddfrowbin_to_si(
    bin_rowmajor_path=row_path,
    metadata = metadata
)

# Extract SpikeInterface recording object and print information
rec_si = conv_si[0]
print(f"Sampling frequency: {rec_si.sampling_frequency}")
print(f"Number of channels: {rec_si.get_num_channels()}")

## 2. Loading SpikeInterface Recordings

SpikeInterface is a popular Python library for extracellular electrophysiology data. NeuRodent can directly use SpikeInterface recordings:

In [3]:
# need pyedflig

# Example: Loading SpikeInterface recordings
si_data_path = Path("../../notebooks/tests/test-data/A10 KO 12_13_2023")

lro_si = core.LongRecordingOrganizer(
    base_folder_path=si_data_path,
    mode="si",  # Change based on your data format
    manual_datetimes=datetime(2023, 12, 12),
    extract_func=se.read_edf,
    stream_id='0',
    input_type='file',
    file_pattern='*.edf',
)

print(f"Number of recordings: {len(lro.colbins)}")
print(f"Sampling frequency: {lro.meta.f_s}")
print(f"Number of channels: {lro.meta.n_channels}")

2026-01-30 16:21:02,336 - INFO - Recording already at target sampling rate (1000 Hz), no resampling needed
2026-01-30 16:21:02,337 - INFO - Finalizing file timestamps
2026-01-30 16:21:02,338 - INFO - Using manual timestamps: 1 file end times specified


Number of recordings: 1
Sampling frequency: 1000
Number of channels: 10


## 3. Loading MNE Objects

MNE-Python is a widely-used library for MEG and EEG analysis. NeuRodent can work with MNE Raw objects:

In [6]:
import mne
from datetime import datetime
from pathlib import Path

data_path = Path("../../notebooks/tests/test-data/A10 KO 12_13_2023")
animal_id = "A10 KO 12_13_2023"


# Create LongRecordingOrganizer with MNE object
lro_mne = core.LongRecordingOrganizer(
    base_folder_path=data_path,
    manual_datetimes=datetime(2023, 12, 13),
    mode="mne",
    extract_func = mne.io.read_raw_fif,
    input_type='file',
    file_pattern = '*.fif',
    
)

print(f"Sampling frequency: {lro_mne.meta.f_s}")
print(f"Number of channels: {lro_mne.meta.n_channels}")

2026-01-30 16:22:10,165 - INFO - Using cached intermediate file: A10 KO 12_13_2023_mne-to-rec.edf
2026-01-30 16:22:10,166 - INFO - Using cached intermediate: A10 KO 12_13_2023_mne-to-rec.edf
2026-01-30 16:22:10,166 - INFO - Loading cached metadata from ../../notebooks/tests/test-data/A10 KO 12_13_2023/A10 KO 12_13_2023_mne-to-rec.edf.meta.json
2026-01-30 16:22:10,169 - INFO - Loaded cached metadata: 10 channels, 1000.0 Hz
2026-01-30 16:22:10,170 - INFO - Reading cached edf file
2026-01-30 16:22:10,177 - INFO - Recording already at target sampling rate (1000 Hz), no resampling needed
2026-01-30 16:22:10,178 - INFO - Finalizing file timestamps
2026-01-30 16:22:10,179 - INFO - Using manual timestamps: 1 file end times specified


Sampling frequency: 1000.0
Number of channels: 10


## 4. Loading NWB Files

Neurodata Without Borders (NWB) is a standardized format for neurophysiology data:

In [ ]:
# Example: Loading NWB files
nwb_path = Path("/path/to/nwb/file.nwb")

# First, load with SpikeInterface's NWB extractor
import spikeinterface.extractors as se

recording_nwb = se.read_nwb(nwb_path)

# Then use with LongRecordingOrganizer
lro_nwb = core.LongRecordingOrganizer(
    base_folder=None,
    animal_id=animal_id,
    mode="si",
    si_recordings=[recording_nwb],
)

print(f"Loaded NWB data with {len(lro_nwb.recordings)} recordings")

## 5. Loading Other Formats

### Neuroscope/Neuralynx

For `.dat` or `.eeg` files:

In [ ]:
# Load using SpikeInterface extractors
neuroscope_path = Path("/path/to/neuroscope/data.dat")
recording_neuroscope = se.read_neuroscope(neuroscope_path)

lro_neuroscope = core.LongRecordingOrganizer(
    base_folder=None,
    animal_id=animal_id,
    mode="si",
    si_recordings=[recording_neuroscope],
)

### Open Ephys

For Open Ephys `.continuous` files:

In [ ]:
# Load using SpikeInterface extractors
openephys_path = Path("/path/to/openephys/folder")
recording_openephys = se.read_openephys(openephys_path)

lro_openephys = core.LongRecordingOrganizer(
    base_folder=None,
    animal_id=animal_id,
    mode="si",
    si_recordings=[recording_openephys],
)

## 6. Inspecting Loaded Data

Once data is loaded, you can inspect its properties:

In [ ]:
# Access recordings
recording = lro.colbins[0]

# Get basic properties
print(f"Sampling frequency: {recording.} Hz")
print(f"Number of channels: {recording.n_channels}")
print(f"Channel IDs: {recording.get_channel_ids()}")
print(f"Duration: {recording.get_num_frames() / recording.get_sampling_frequency()} seconds")

# Get channel metadata
channel_locations = recording.get_channel_locations()
print(f"Channel locations: {channel_locations}")

AttributeError: 'str' object has no attribute 'f_s'

## 7. Working with Multiple Recordings

NeuRodent can handle multiple recordings from the same animal (e.g., different sessions or days):

In [ ]:
# Example: Loading multiple recordings
data_folder = Path("/path/to/multi/session/data")

lro_multi = core.LongRecordingOrganizer(
    base_folder=data_folder,
    animal_id=animal_id,
    mode="bin",
)

print(f"Total recordings: {len(lro_multi.recordings)}")

# Iterate through recordings
for i, recording in enumerate(lro_multi.recordings):
    duration = recording.get_num_frames() / recording.get_sampling_frequency()
    print(f"Recording {i}: {duration:.1f} seconds")

## 8. Metadata and Time Information

NeuRodent extracts metadata from filenames and paths, including timing information:

In [ ]:
from neurodent.core import is_day

# Check if recording is during day or night
# (assuming timestamp in filename or metadata)
example_timestamp = "2023-12-15_14-30-00"  # Example: 2:30 PM

is_daytime = is_day(example_timestamp)
print(f"Recording at {example_timestamp} is {'day' if is_daytime else 'night'}time")

# Access metadata from recordings
for recording in lro_bin.recordings:
    metadata = recording.get_property("metadata") if recording.has_property("metadata") else None
    if metadata:
        print(f"Recording metadata: {metadata}")

## 9. Advanced: Custom Data Loading

For custom formats, you can create SpikeInterface Recording objects and pass them to `LongRecordingOrganizer`:

In [ ]:
import spikeinterface as si

# Example: Create a recording from numpy array
# (useful for custom formats or testing)
num_channels = 16
sampling_frequency = 1000  # Hz
duration = 60  # seconds
num_samples = int(sampling_frequency * duration)

# Generate random data (replace with your actual data)
data = np.random.randn(num_channels, num_samples)

# Create SpikeInterface recording
recording_custom = si.NumpyRecording(
    traces_list=[data],
    sampling_frequency=sampling_frequency,
)

# Set channel IDs
channel_ids = [f"CH{i:02d}" for i in range(num_channels)]
recording_custom = recording_custom.rename_channels(
    new_channel_ids=channel_ids
)

# Use with LongRecordingOrganizer
lro_custom = core.LongRecordingOrganizer(
    base_folder=None,
    animal_id=animal_id,
    mode="si",
    si_recordings=[recording_custom],
)

print("Custom recording created successfully!")

## Summary

In this tutorial, you learned:

1. How to load data from multiple formats (binary, SpikeInterface, MNE, NWB, etc.)
2. How to inspect loaded data properties
3. How to work with multiple recordings
4. How to handle metadata and timing information
5. How to create custom recordings for non-standard formats

## Next Steps

- **[Basic Usage Tutorial](basic_usage.ipynb)**: Complete workflow from loading to visualization
- **[Windowed Analysis Tutorial](../tutorials/windowed_analysis.ipynb)**: Extract features from loaded data
- **[Spike Analysis Tutorial](../tutorials/spike_analysis.ipynb)**: Work with spike-sorted data

## Summary

In this tutorial, you learned:

1. How to load data from multiple formats (binary, SpikeInterface, MNE, NWB, etc.)
2. How to inspect loaded data properties
3. How to work with multiple recordings
4. How to handle metadata and timing information
5. How to create custom recordings for non-standard formats

## Next Steps

- **[Basic Usage Tutorial](basic_usage.ipynb)**: Complete workflow from loading to visualization
- **[Windowed Analysis Tutorial](../tutorials/windowed_analysis.ipynb)**: Extract features from loaded data
- **[Spike Analysis Tutorial](../tutorials/spike_analysis.ipynb)**: Work with spike-sorted data